<a href="https://colab.research.google.com/github/charansai12322/skillmapagentwithmcp/blob/main/mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-google-genai
!pip install langchain langchain-tavily
!pip install langchain-mcp-adapters

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [3]:
google_api_key = userdata.get('sec1')
model = init_chat_model("google_genai:gemini-3.5-flash", api_key=google_api_key)

In [4]:
from langchain_tavily import TavilySearch
from google.colab import userdata

In [5]:
tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

In [6]:
import requests
from langchain.tools import tool
from google.colab import userdata

In [8]:
@tool
def search_jobs(skill: str, location: str) -> list:
  """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
  print(f"\nCalling search_jobs tool")
  print(f"Searching jobs for: {skill} in {location}")

  rapidapi_key = userdata.get('RAPIDAPI_KEY')

  url = "https://jsearch.p.rapidapi.com/search"
  headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }
  querystring = {
    "query": f"{skill} in {location}",
    "page": "1",
    "country": "in",
    "employment_types": "INTERN,FULLTIME",
    "job_requirements": "no_experience,under_3_years_experience"
  }
  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()
  jobs = data.get("data", [])
  print(f"Found {len(jobs)} jobs\n")

  result = []
  for job in jobs:
    result.append({
      "title": job.get("job_title"),
      "company": job.get("employer_name"),
      "location": job.get("job_city"),
      "apply_link": job.get("job_apply_link")
    })
  return result


In [9]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- search tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""



In [14]:
client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": "https://connect.composio.dev/mcp",
            "headers": {
                "x-consumer-api-key": "COMPOSIO_API_KEY"
            },
        }
    }
)

In [20]:

from google.colab import userdata
from langchain_mcp_adapters.client import MultiServerMCPClient

composio_key = userdata.get("COMPOSIO_API_KEY")

if not composio_key:
    raise ValueError(
        "Add COMPOSIO_CONSUMER_API_KEY to Colab Secrets "
        "and enable notebook access."
    )

client = MultiServerMCPClient({
    "mcp_tavily": {
        "transport": "http",
        "url": "https://connect.composio.dev/mcp",
        "headers": {
            "x-consumer-api-key": composio_key
        }
    }
})

print("Composio MCP client configured.")


Composio MCP client configured.


In [11]:
from langchain.agents import create_agent

In [23]:

import asyncio
import traceback

async def skill_map_agent():
    try:
        # Step 1: Get tools from the MCP server
        print("Connecting to MCP server...")

        mcp_tools = await client.get_tools()

        print(f"Successfully loaded {len(mcp_tools)} MCP tools.")

        # Step 2: Combine MCP tools with custom tools
        all_tools = mcp_tools + [search_jobs]

        print(f"Total tools available: {len(all_tools)}")

        # Step 3: Create the agent
        agent = create_agent(
            model=model,
            tools=all_tools,
            system_prompt=system_prompt,
            debug=True
        )

        # Step 4: Define the user query
        user_query = "What's the demand for generative AI in the industry?"

        # Step 5: Invoke the agent
        print("\nRunning Skill Map Agent...\n")

        response = await agent.ainvoke({
            "messages": [
                {
                    "role": "user",
                    "content": user_query
                }
            ]
        })

        # Step 6: Print the final response safely
        print("\n" + "=" * 50)
        print("SKILL MAP AGENT RESPONSE")
        print("=" * 50)

        messages = response.get("messages", [])

        if messages:
            final_message = messages[-1]
            content = final_message.content

            if isinstance(content, str):
                print(content)

            elif isinstance(content, list):
                for block in content:
                    if isinstance(block, dict):
                        if block.get("type") == "text":
                            print(block.get("text", ""))
                    elif hasattr(block, "text"):
                        print(block.text)
                    else:
                        print(block)

            else:
                print(content)
        else:
            print("No messages were returned by the agent.")
            print(response)

    except Exception as e:
        print("\nERROR OCCURRED")
        print("=" * 50)
        traceback.print_exception(e)

# Run the async function in Google Colab
await skill_map_agent()


Connecting to MCP server...


Successfully loaded 11 MCP tools.
Total tools available: 12

Running Skill Map Agent...

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry?", additional_kwargs={}, response_metadata={}, id='a19e7e86-e877-41b1-94e2-a1778932e65e')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'COMPOSIO_SEARCH_TOOLS', 'arguments': '{"session": {"generate_id": true}, "queries": [{"known_fields": "topic:Generative AI", "use_case": "search for industry demand salary insights and career trends for Generative AI"}]}'}, '__gemini_function_call_thought_signatures__': {'call_1976289': 'EoMMCoAMAWkUfRNkJZmqUfAjlNKA9q/5Ik/WTmHNts6tNotNvK+GHCyqnHIM28hB1nBKS6Py/vaY9m1MfeJyVp4S2yHP/DxPqLtBpA4oj0TwZGGdvOCYe4GsHJanKt5zmuoZ0Q/wTct+i0+LCR/LWde3Zg1lLWeMdAb5gBZpl7qMarRh6+VWQmAhmJ0PfnzUF6JXKGcyvDr4KOW40L9IhBsmDoT12E2us7Etx0BZRO6NGSH9cR2IAOvD0pRu+sJUINznKi6wfl7CbU4rxnG9KdXNKhoqqUIoUbZnihQ7IqnwcVayRfw91vMBO8OVkG5JoKRW+lVnGrV

[updates] {'tools': {'messages': [ToolMessage(content=[{'type': 'text', 'text': '{"successful":true,"data":{"results":[{"index":1,"use_case":"search for industry demand salary insights and career trends for Generative AI","execution_guidance":"IMPORTANT: Follow the recommended plan below. Extract steps before execution, adapt to your current context, execute sequentially with current_step parameter, and check known pitfalls to avoid errors.","recommended_plan_steps":["[Required] [Step]: Gather foundational, attributable sources using COMPOSIO_SEARCH_WEB (focus on forward-looking trends; capture results.citations/data.citations and dedupe by canonical URL).","[Optional (if recency-sensitive coverage is needed)] [Step]: Add timely reporting using COMPOSIO_SEARCH_NEWS (run a few query variants; retain link/source/published_at/snippet and dedupe by news_results[].link).","[Optional (if stronger technical evidence is needed)] [Step]: Locate papers/benchmarks using COMPOSIO_SEARCH_SCHOLAR (c

Found 0 jobs

[updates] {'tools': {'messages': [ToolMessage(content=[], name='search_jobs', id='fedc3748-b50d-4fc3-92d3-de465e65dcd0', tool_call_id='call_2613070')]}}
[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry?", additional_kwargs={}, response_metadata={}, id='a19e7e86-e877-41b1-94e2-a1778932e65e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'COMPOSIO_SEARCH_TOOLS', 'arguments': '{"session": {"generate_id": true}, "queries": [{"known_fields": "topic:Generative AI", "use_case": "search for industry demand salary insights and career trends for Generative AI"}]}'}, '__gemini_function_call_thought_signatures__': {'call_1976289': 'EoMMCoAMAWkUfRNkJZmqUfAjlNKA9q/5Ik/WTmHNts6tNotNvK+GHCyqnHIM28hB1nBKS6Py/vaY9m1MfeJyVp4S2yHP/DxPqLtBpA4oj0TwZGGdvOCYe4GsHJanKt5zmuoZ0Q/wTct+i0+LCR/LWde3Zg1lLWeMdAb5gBZpl7qMarRh6+VWQmAhmJ0PfnzUF6JXKGcyvDr4KOW40L9IhBsmDoT12E2us7Etx0BZRO6NGSH9cR2IAOvD0pRu+sJUINznKi6wfl7CbU4rxnG9KdXNKhoqqUIo

Found 0 jobs

[updates] {'tools': {'messages': [ToolMessage(content=[], name='search_jobs', id='130db624-c0cb-492d-8c97-e2fa0ed49d8e', tool_call_id='call_1692771')]}}
[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry?", additional_kwargs={}, response_metadata={}, id='a19e7e86-e877-41b1-94e2-a1778932e65e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'COMPOSIO_SEARCH_TOOLS', 'arguments': '{"session": {"generate_id": true}, "queries": [{"known_fields": "topic:Generative AI", "use_case": "search for industry demand salary insights and career trends for Generative AI"}]}'}, '__gemini_function_call_thought_signatures__': {'call_1976289': 'EoMMCoAMAWkUfRNkJZmqUfAjlNKA9q/5Ik/WTmHNts6tNotNvK+GHCyqnHIM28hB1nBKS6Py/vaY9m1MfeJyVp4S2yHP/DxPqLtBpA4oj0TwZGGdvOCYe4GsHJanKt5zmuoZ0Q/wTct+i0+LCR/LWde3Zg1lLWeMdAb5gBZpl7qMarRh6+VWQmAhmJ0PfnzUF6JXKGcyvDr4KOW40L9IhBsmDoT12E2us7Etx0BZRO6NGSH9cR2IAOvD0pRu+sJUINznKi6wfl7CbU4rxnG9KdXNKhoqqUIo

Found 0 jobs

[updates] {'tools': {'messages': [ToolMessage(content=[], name='search_jobs', id='4d52603d-5fea-4f51-9163-29cb179f82b2', tool_call_id='call_1810198')]}}
[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry?", additional_kwargs={}, response_metadata={}, id='a19e7e86-e877-41b1-94e2-a1778932e65e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'COMPOSIO_SEARCH_TOOLS', 'arguments': '{"session": {"generate_id": true}, "queries": [{"known_fields": "topic:Generative AI", "use_case": "search for industry demand salary insights and career trends for Generative AI"}]}'}, '__gemini_function_call_thought_signatures__': {'call_1976289': 'EoMMCoAMAWkUfRNkJZmqUfAjlNKA9q/5Ik/WTmHNts6tNotNvK+GHCyqnHIM28hB1nBKS6Py/vaY9m1MfeJyVp4S2yHP/DxPqLtBpA4oj0TwZGGdvOCYe4GsHJanKt5zmuoZ0Q/wTct+i0+LCR/LWde3Zg1lLWeMdAb5gBZpl7qMarRh6+VWQmAhmJ0PfnzUF6JXKGcyvDr4KOW40L9IhBsmDoT12E2us7Etx0BZRO6NGSH9cR2IAOvD0pRu+sJUINznKi6wfl7CbU4rxnG9KdXNKhoqqUIo

Found 0 jobs

[updates] {'tools': {'messages': [ToolMessage(content=[], name='search_jobs', id='5297c12d-1b31-497d-b5ca-a829792e7a7f', tool_call_id='call_1453534')]}}
[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry?", additional_kwargs={}, response_metadata={}, id='a19e7e86-e877-41b1-94e2-a1778932e65e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'COMPOSIO_SEARCH_TOOLS', 'arguments': '{"session": {"generate_id": true}, "queries": [{"known_fields": "topic:Generative AI", "use_case": "search for industry demand salary insights and career trends for Generative AI"}]}'}, '__gemini_function_call_thought_signatures__': {'call_1976289': 'EoMMCoAMAWkUfRNkJZmqUfAjlNKA9q/5Ik/WTmHNts6tNotNvK+GHCyqnHIM28hB1nBKS6Py/vaY9m1MfeJyVp4S2yHP/DxPqLtBpA4oj0TwZGGdvOCYe4GsHJanKt5zmuoZ0Q/wTct+i0+LCR/LWde3Zg1lLWeMdAb5gBZpl7qMarRh6+VWQmAhmJ0PfnzUF6JXKGcyvDr4KOW40L9IhBsmDoT12E2us7Etx0BZRO6NGSH9cR2IAOvD0pRu+sJUINznKi6wfl7CbU4rxnG9KdXNKhoqqUIo

[updates] {'tools': {'messages': [ToolMessage(content=[{'type': 'text', 'text': '{"successful":true,"data":{"results":[{"response":{"successful":true,"data":{"answer":"Several companies are currently hiring for Generative AI engineering roles. Accenture Federal Services has openings for a Generative AI Applications Engineer [1] and a GenAI Engineer [2], both requiring U.S. citizenship and experience with RAG and LLM orchestration. Figureai is seeking a Helix AI Engineer to train and deploy large-scale generative models for robotics [3]. Other active listings include a Senior GenAI Engineer at Pigment [4], a Lead AI Engineer at Egen [5], and a Software Engineer with Gen AI expertise at Pipedrive [6]. Jobgether is also hiring for full-stack and pipeline engineering roles [7][8].","citations":[{"id":"https://boards.greenhouse.io/accenturefederalservices/jobs/4669273006?gh_jid=4669273006","publishedDate":"2026-04-01T00:00:00.000Z","title":"Generative AI Applications Engineer (Agents & RAG)